# Pronóstico de 2025 con decaimiento post-pico

El tablero público mide el segundo semestre de 2025 (semanas 27 a 52). El privado, el que define al ganador, mide el primero. El envío anterior (2048,50) se pasó en el interior: la mediana de 2019 a 2024 mezcla dos olas. Este cuaderno deja caer, semana a semana, los departamentos agotados después de un pico nacional, y solo suma un modelo del zoo si el walk-forward baja.

El código que entrena está en `pipeline.py`, al lado de este cuaderno. Las celdas leen el resultado ya calculado y muestran la compuerta de las redes.


## Datos

La historia llega hasta 2024. 2025 trae el clima y no trae casos. El archivo de envío se escribe en esta carpeta, con las mismas columnas y el mismo orden que `sample_submission.csv`.


In [1]:
import json
from pathlib import Path

import pandas as pd

aqui = Path.cwd().resolve()
raiz = aqui
for carpeta in [aqui, *list(aqui.parents)[:6]]:
    if (carpeta / "data" / "train.parquet").exists():
        raiz = carpeta
        break
ensayo = raiz / "ensayos" / "juan" / "2026-11-03-zoo-bayesiano"
resultado = json.loads((ensayo / "resultados.json").read_text(encoding="utf-8"))
train = pd.read_parquet(raiz / "data" / "train.parquet")
test = pd.read_parquet(raiz / "data" / "test.parquet")
print(train.shape, test.shape)
print("dispositivo", resultado["dispositivo"])
print("anios", int(train.anio.min()), int(train.anio.max()), "departamentos", train.id_departamento.nunique())


(29140, 19) (1612, 18)
dispositivo cuda
anios 2007 2024 departamentos 31


## Piso de años bajos y decaimiento

`b_low2_4` es la media suavizada de los dos años más bajos de los últimos cuatro. `b_low3_6` hace lo mismo con los tres más bajos de los últimos seis. El piso es el mínimo de las dos. El decaimiento solo corre si el año anterior fue un pico nacional (media por encima de 45 y primer semestre al menos 15 % arriba del segundo): 2011, 2017, 2021 y 2025.

En esos años, un departamento que ya venía bajando después de dos temporadas altas, o de un solo año extremo, se acerca al piso con `1 - exp(-semana / tau)`. La costa que creció de verdad se mezcla, solo en el segundo semestre, con el máximo entre la base reciente y los dos años bajos de los últimos cuatro. Optuna busca tau, la mezcla y los umbrales. No usa el score de 2025.


In [2]:
print("referencia tau 14", resultado["referencia"])
print("optuna", resultado["optuna"])
print("elegido", resultado["decaimiento"])
print(pd.DataFrame(resultado["anios"]).to_string(index=False))


referencia tau 14 {'mod_h1': 2161.37, 'mod_h2': 2285.44, 'post_h1': 2761.36, 'post_h2': 2490.04}
optuna {'tau': 12.39, 'mezcla_costa': 0.5996, 'umbral_par': 1.8324, 'umbral_mega': 5.7021, 'tope_q4': 1.0824}
elegido {'mod_h1': 2159.49, 'mod_h2': 2283.22, 'post_h1': 2758.07, 'post_h2': 2486.15}
 anio       h1      h2
 2014   1418.9  2067.6
 2015   2647.2  2100.1
 2016   9742.7  8117.5
 2017    645.7   617.1
 2018    415.6  1504.6
 2019   4886.2  7714.1
 2020   8399.0  3326.2
 2021    568.6  3933.7
 2022   1021.4  2433.2
 2023   5108.6 19464.1
 2024 152277.7 65749.4


## Zoo

Cinco redes en PyTorch, en GPU si hay CUDA: MLP, RNN, LSTM, GRU y TCN. La cabeza es una compuerta softmax sobre las bases causales, más un término de clima centrado en cero, así que la predicción no se sale de la escala. LightGBM, XGBoost de profundidad 3 y Ridge sin intercepto se entrenan aparte para el primer semestre y para el segundo. Cada año de validación usa solo 2010 hasta el año anterior, y deja fuera los años de brote del propio departamento.

La tabla de abajo mezcla un 30 % de cada modelo con el decaimiento. Sirve para comparar. El peso que entra al archivo lo decide otra búsqueda, una por tramo (semanas 1–13, 14–26 y 27–52), y solo se queda si baja los años moderados en ese tramo sin empeorar los años post-pico.


In [3]:
print(pd.DataFrame(resultado["comparacion"]).to_string(index=False))
for tramo in ("pesos_q1", "pesos_q2", "pesos_h2"):
    activos = {nombre: peso for nombre, peso in resultado[tramo].items() if peso > 0.01}
    print(tramo, activos)
print("mezcla final", resultado["mezcla"])


     modelo  mod_h1  mod_h2  post_h1  post_h2
decaimiento  2159.5  2283.2   2758.1   2486.1
      ridge  2241.2  2455.8   2934.4   2824.1
   lightgbm  2131.2  2317.8   2795.9   2546.0
    xgboost  2187.0  2288.8   2833.9   2498.8
        mlp  2258.0  2367.9   2921.9   2593.0
        rnn  2245.3  2309.9   2901.3   2520.2
       lstm  2223.1  2305.3   2866.2   2509.8
        gru  2249.7  2315.7   2910.4   2528.4
        tcn  2253.0  2332.1   2927.6   2558.6
pesos_q1 {'decaimiento': 1.0}
pesos_q2 {'decaimiento': 1.0}
pesos_h2 {'decaimiento': 1.0}
mezcla final {'mod_h1': 2159.49, 'mod_h2': 2283.22, 'post_h1': 2758.07, 'post_h2': 2486.15}


## Compuerta de las redes

La convolución del TCN rellena solo por la izquierda, así que el paso de una semana no ve las siguientes. La mezcla con las bases está acotada: como mucho un 35 % sale del prior, y el clima mueve ese nivel en un 15 % hacia arriba o hacia abajo.


In [4]:
import inspect
import sys

sys.path.insert(0, str(ensayo))
import pipeline

print(inspect.getsource(pipeline.decaer))
print(inspect.getsource(pipeline.RedZoo))


def decaer(marco: pd.DataFrame, previo: np.ndarray, tau: float, mezcla_costa: float, umbral_par: float, umbral_mega: float, tope_q4: float) -> np.ndarray:
    if float(marco.nat_post_peak.iloc[0]) < 0.5:
        return previo
    semana = marco.semana.to_numpy(float)
    r1 = marco.r1_clean.to_numpy(float)
    r2 = marco.r2_clean.to_numpy(float)
    cae = (marco.q4_over_h1.to_numpy(float) < tope_q4) & (((r1 > umbral_par) & (r2 > umbral_par)) | (r1 > umbral_mega))
    piso = marco.b_trough.to_numpy(float)
    moderno = marco.b_modern.to_numpy(float)
    crece = marco.b_low2_4.to_numpy(float) > 1.3 * marco.b_med_all.to_numpy(float)
    peso_interior = 1.0 - np.exp(-semana / tau)
    peso_costa = np.where((semana >= 27) & crece, mezcla_costa, 0.0)
    return np.where(cae, (1.0 - peso_interior) * previo + peso_interior * piso, (1.0 - peso_costa) * previo + peso_costa * moderno)

class RedZoo(nn.Module):
    """A bounded gate over causal bases, plus a climate term centered at zero."""

    

## Archivo

Si un tramo del zoo no gana el walk-forward, ese tramo queda en el decaimiento. Alpha es la proyección del segundo semestre sobre la recta que une los dos envíos ya calificados. El valor exacto del tablero es 0,794. Sirve para ver que el archivo no se pasó del segundo envío. No es el score público: los departamentos del interior se mueven fuera de esa recta.


In [5]:
envio = pd.read_csv(ensayo / "submission.csv")
muestra = pd.read_csv(raiz / "data" / "sample_submission.csv")
print("filas", len(envio), "ids", bool(envio.id.equals(muestra.id)))
print("nulos", int(envio.dengue.isna().sum()), "negativos", int((envio.dengue < 0).sum()))
print("media", resultado["media"], "h1", resultado["media_h1"], "h2", resultado["media_h2"])
print("alpha", resultado["alpha_h2"])
print(pd.DataFrame(resultado["niveles_h2"]).head(12).to_string(index=False))


filas 1612 ids True
nulos 0 negativos 0
media 45.04 h1 51.12 h2 38.96
alpha 0.8222
      departamento    p1    p2  dengue
           Bolivar  26.3 221.9   202.3
   Valle Del Cauca 147.2 192.9   117.5
         Atlantico  50.5 164.5   163.1
              Meta  77.6 129.8    58.8
           Cordoba  31.9 121.5    99.6
            Tolima  66.9 121.1    52.1
         Antioquia  77.3 115.4    36.6
Norte De Santander  78.6 103.6    50.1
         Santander 201.1  99.6    33.3
             Sucre  35.2  98.3    88.7
             Huila  65.5  62.4    22.6
             Cesar  47.6  59.4    59.4
